# Retrain the thyroid image model on TN5000 (official split)

Trains EfficientNet-B0 to classify thyroid ultrasound images as **benign** or **malignant**.

* Train on TN5000's official **train** split (3,500 images), pick the best epoch on **val** (500), score the untouched **test** split (1,000) once.
* Class-weighted loss (the data is ~70% malignant) and ImageNet normalisation.
* Outputs: `cancer_efficientnet.pth`, `cancer_efficientnet.json` (preprocessing + metrics), `metrics.json`, `roc.png`, `confusion_matrix.png`, `history.csv`.

**Before you run:** `Runtime → Change runtime type → T4 GPU`, and put `TN5000_forReview.zip` in the top level of your Google Drive (*My Drive*).

Dataset: Zhang et al., *TN5000: An Ultrasound Image Dataset for Thyroid Nodule Detection and Classification*, Scientific Data 12, 1437 (2025).

In [ ]:
# 1. Mount Google Drive and copy the dataset to the fast local disk
from google.colab import drive
drive.mount('/content/drive')

ZIP_IN_DRIVE = '/content/drive/MyDrive/TN5000_forReview.zip'   # change if you put it elsewhere
!cp "$ZIP_IN_DRIVE" /content/TN5000.zip && ls -lh /content/TN5000.zip

In [ ]:
# 2. Get the project code
!rm -rf /content/thyroid-ml-system
!git clone -q https://github.com/lonewolf1621/thyroid-ml-system /content/thyroid-ml-system
%cd /content/thyroid-ml-system
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (enable a GPU runtime!)')

In [ ]:
# 3. Train (about 10-15 minutes on a T4 GPU)
!python src/train_image_model.py --data /content/TN5000.zip --out /content/outputs --epochs 15 --batch-size 32

In [ ]:
# 4. Results on the held-out test set
import json
from IPython.display import Image, display
m = json.load(open('/content/outputs/metrics.json'))['test']
for k in ['accuracy','balanced_accuracy','sensitivity','specificity','precision','f1','auc','majority_class_baseline']:
    print(f'{k:>24}: {m[k]:.3f}')
print('confusion matrix:', m['confusion_matrix'])
display(Image('/content/outputs/roc.png'))
display(Image('/content/outputs/confusion_matrix.png'))

In [ ]:
# 5. Save everything to Google Drive (folder: My Drive/thyroid_model_v2)
!mkdir -p /content/drive/MyDrive/thyroid_model_v2
!cp /content/outputs/* /content/drive/MyDrive/thyroid_model_v2/
!ls -lh /content/drive/MyDrive/thyroid_model_v2